<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Collecte de données

Les modèles de détection d'objets basés sur l'apprentissage automatique peuvent s'avérer extrêmement utiles. Ils sont plus rapides et souvent plus fiables que les modèles de vision par ordinateur traditionnels. De plus, l'utilisation de poids de modèles pré-entraînés permet de réduire considérablement le temps d'entraînement.

Voici un exemple de résultat possible pour un détecteur d'objets :


<iframe width="800" height="500"
src="https://www.youtube.com/embed/3jD02dxL6gg" 
frameborder="0" 
allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" 
allowfullscreen
style="margin: auto; display: block"></iframe>


Vous allez créer votre propre base de données de détection d'objets pour Duckietown. Vous découvrirez la structure générale que doit suivre un tel jeu de données. Vous entraînerez le modèle de détection d'objets sur ce jeu de données [dans un notebook suivant](../03-Training/training.ipynb), puis vous exporterez le modèle de PyTorch vers ONNX pour faciliter son déploiement sur différentes plateformes. Enfin, vous intégrerez le modèle et testerez l'intégration [dans le dernier notebook](../04-Integration/integration.ipynb), afin que votre Duckiebot sache reconnaître les piétons canards (et donc les éviter). Vous pourrez tester votre détecteur d'objets en simulation et sur votre véritable Duckiebot.



## Mise en place

Tout d'abord, nous avons besoin de variables globales. Celles-ci vous permettent de modifier le répertoire où sont stockées les données nécessaires. Vous pouvez également modifier la taille de l'image en fonction de votre modèle final, mais vous pourrez vous en préoccuper plus tard.

In [ ]:
DATA_DIR="../../assets/data/"
# Il s'agit du pourcentage de données qui seront intégrées à l'ensemble d'entraînement (par opposition à l'ensemble de test).
TRAIN_TEST_SPLIT_PERCENTAGE = 0.8

Nous décrirons le processus global que vous suivrez dans la [prochaine section](#flux-de-travail). Ensuite, vous créerez votre propre ensemble de données avec des images simulées et réelles dans la section de [collecte de données](#collecte).

## Flux de travail d'entraînement du modèle <a id="flux-de-travail"></a>

À quoi ressemble un jeu de données de détection d'objets ? Bien entendu, les détails dépendent des conventions utilisées par chaque modèle, mais l'idée générale est intuitive :

- Nous avons besoin d'une **image**.

- Chaque image peut contenir **plusieurs objets**.

- Pour chaque objet, nous avons besoin :

   - d'un **cadre de délimitation** (indiquant sa position)
   - d'une **étiquette de classe** (indiquant sa nature)
### Format des boîtes englobantes

Il existe plusieurs manières courantes de représenter les boîtes englobantes :

- `x_min y_min width height`
- `x_min y_min x_max y_max`
- `x_center y_center width height`

Dans cet exercice, nous utiliserons un modèle de type YOLO (YOLOv11 de la bibliothèque [Ultralytics](https://github.com/ultralytics/ultralytics)), qui attend des boîtes englobantes au format :
> `x_center y_center width height`

Les quatre valeurs sont normalisées entre 0 et 1 par rapport à la largeur et à la hauteur de l'image. Chaque fichier d'étiquettes (.txt) contient une ligne par objet.


![image of a bounding box](../../assets/images/bbox.png)



### Comment obtenir des boîtes englobantes

Dans un processus traditionnel de détection d'objets, il faudrait étiqueter un ensemble de données d'images **manuellement**. Cependant, cette tâche étant fastidieuse et les images que nous allons collecter étant relativement simples, nous allons générer automatiquement les étiquettes à l'aide de [**SAM3**](https://github.com/facebookresearch/sam3), un modèle de base capable de produire des annotations de haute qualité pour les objets dans une image ou une vidéo. Nous l'utiliserons uniquement pour étiqueter les images collectées lors de l'étape de collecte des données.

Pour en savoir plus sur l'obtention des boîtes englobantes, vous pouvez consulter le [tutoriel](https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html) sur la détection d'objets avec PyTorch.

Le processus d'étiquetage automatique fonctionne comme suit :

1. **Collecte des images RVB**

   Les images proviennent du robot (virtuel ou réel).

2. **Exécution de SAM3 pour obtenir les boîtes englobantes des instances**

   SAM3 prédit un ensemble de boîtes englobantes dans l'image en fonction des indications ou des classes prédéfinies. Chaque boîte englobante correspond à un objet visuellement distinct.

   Ceci produit les coordonnées `(x_min, y_min, x_max, y_max)` en pixels, que nous traitons ensuite pour correspondre au format attendu par **Ultralytics** pour l'entraînement.

3. **Classes d'intérêt à Duckietown**

   L'objectif de cet exercice est de rendre Duckietown plus sûr : nous voulons pouvoir détecter les piétons déguisés en canards sur la route et éviter de les écraser. Voici quelques classes courantes qui pourraient vous intéresser :

   - canard
   - cône
   - camion
   - bus
   - Duckiebot

   Vous devrez configurer une invite pour les classes qui vous intéressent. Pour les autres exercices, nous nous concentrerons uniquement sur la détection des canards, mais vous pouvez créer votre détecteur avec autant d'objets que nécessaire.

4. **Convertir les boîtes au format YOLO**
  
   Nous avons fourni une fonction pour créer un jeu de données à partir des étiquettes collectées et des images afin de vous permettre d'entraîner votre détecteur de duckie une fois que l'étape d'auto-identification est complétée.

Dans les sections suivantes, nous expliquerons les étapes de génération des données.

## Collecte de données <a id="collecte"></a>

### Outils et format


En mode `data_collection`, l'agent enregistre les données de la caméra en continu pendant que vous pilotez le robot.

Le format de votre jeu de données dépend fortement de votre modèle. Ici, vous utiliserez un modèle [Ultralytics](https://github.com/ultralytics/ultralytics). Suivez donc attentivement leur [guide d'entraînement avec des données personnalisées](https://docs.ultralytics.com/modes/train/). Vos données doivent respecter la structure de répertoires suivante :

![image du format d'enregistrement du jeu de données](../../assets/images/dataset_format.png)

Le jeu de données sera nommé `duckietown_dataset` et sera stocké dans le répertoire `/data` de votre robot, puis copié dans le répertoire [assets/data](../../assets/data/) de ce repositoire.

Nous créerons deux sous-répertoires dans ce dossier : `train` et `val`. Dans chacun de ces répertoires, nous allons créer deux sous-répertoires : `images` et `labels`. Dans `images`, vous devez placer vos images, et dans `labels`, les données de leurs boîtes englobantes. Notez que les fichiers d'étiquettes portent le même nom que leurs fichiers image correspondants, mais avec une extension différente. Autrement dit, les étiquettes de `0.jpg` se trouvent dans `0.txt`.

Le format des fichiers d'étiquettes est assez simple. Pour chaque boîte englobante de l'image correspondante, écrivez une ligne de la forme `class x_center y_center width height`. N'oubliez pas que les données en pixels doivent être normalisées entre 0 et 1 (c'est-à-dire que vous pouvez calculer les valeurs habituelles de `x_center y_center width height` en pixels et diviser par la taille de l'image). Par exemple :

    0 0.5 0.5 0.2 0.2
    1 0.60 0.70 0.4 0.2

Ce texte indique : « Un canard (classe 0) est centré dans l’image ; sa largeur et sa hauteur représentent 20 % de celles de l’image. Un cône (classe 1) est également présent ; son centre se situe à 60 % de la valeur maximale de l’axe x et à 70 % de la valeur maximale de l’axe y ; sa largeur représente 40 % de celle de l’image et sa hauteur 20 %.»

Il est recommandé de consulter le guide disponible sur Ultralytics : [guide sur l’entraînement avec des données personnalisées](https://docs.ultralytics.com/modes/train/).

### Exécution de la collecte de données

Nous utilisons désormais le lanceur `data_collection` fourni pour collecter des données en environnement simulé et réel. Ce lanceur exécute l'agent en mode de collecte de données, enregistrant automatiquement les images de la caméra dans le répertoire de données pendant vos déplacements.

*Si vous utilisez un robot virtuel* : assurez-vous d'avoir créé un Duckiebot virtuel et démarré la Duckiematrix en suivant la procédure décrite dans le fichier [README](../../README.md).

Tout d'abord, compilons le code :

    ```shell
    dts code build -R ROBOT_NAME
    ```


Nous pouvons ensuite exécuter le code et spécifier la mode `data_collection` :

    ```shell
    dts code workbench [-m] -R ROBOT_NAME -L data_collection
    ```

Cela lance l'agent de collecte de données. Si vous utilisez Duckiematrix, vous devez ajouter l'option `-m`.

Vous pouvez ouvrir le joystick avec
    
    ```shell
    dts duckiebot keyboard_control ROBOT_NAME
    ```

et pilotez le robot tout en obtenant de nombreuses bonnes vues des canards (Dans Duckiematrix, vous pouvez également y parvenir avec les touches `w`, `a`, `s` et `d` directement dans la fenêtre Duckiematrix comme décrit dans le [README](../../README.md) si vous préférez).

L'agent effectuera les opérations suivantes :

- capture jusqu'à 1 000 images RGB,
- enregistrement dans le répertoire `data/data_collection/`,
- puis arrêt automatique de la collecte d'images.

Vous pouvez également l'arrêter manuellement à tout moment avec `Ctrl+C`.


### Copiez l'ensemble de données sur votre machine locale.

Les images collectées sont stockées sur votre robot dans le répertoire `/data/data_collection/`. Nous devons les copier sur cette machine afin de pouvoir les étiqueter et les utiliser pour l'entraînement.

Dans la cellule ci-dessous, indiquez le nom d'hôte de votre robot dans `ROBOT_NAME`, puis exécutez la cellule pour copier les données via `scp`. Si un mot de passe vous est demandé, utilisez `quackquack`.

> **Astuce :** Vous pouvez également consulter les images collectées via le tableau de bord, à l'adresse `ROBOT_NAME.local` → Gestionnaire de fichiers → `/data/data_collection/`, avant de les copier.

vous devez les copier en exécutant

```bash
scp -r duckie@{ROBOT_NAME}.local:/data/data_collection/. assets/data/data_collection/
```
dans un terminal sur votre ordinateur local (pwd = quackquack)


Une fois la copie terminée, les fichiers `.png` devraient apparaître dans le répertoire `assets/data/data_collection/` de l'explorateur de fichiers situé à gauche.

### En combinant les ensembles de données et les divisions en ensembles d'entraînement et de test

Lors de l'entraînement de modèles d'apprentissage supervisé, il est crucial de se prémunir contre le surapprentissage. Si vous pouvez exclure une partie de vos données d'entraînement, vous pouvez l'utiliser pour vérifier que votre modèle ne surapprend pas en le testant sur ces données exclues. Cet ensemble de données est appelé « ensemble de validation ».

Vous pouvez expérimenter avec la variable `TRAIN_TEST_SPLIT_PERCENTAGE` définie en haut de ce notebook. Ajustez sa valeur pour modifier le pourcentage de données utilisées pour l'entraînement par rapport aux tests.

Une fois le lancement de la collecte de données terminé, vous disposerez d'un seul fichier `data_collection` contenant toutes les images RGB capturées. Avant l'étiquetage automatique de ces images avec SAM3, il est nécessaire de les diviser en sous-ensembles d'**entraînement** et de **validation** au format attendu par YOLO.

In [ ]:
import os
import random
from pathlib import Path
import shutil

def create_train_val_split(data_collection_dir, dataset_dir, split_percentage):
    data_collection_dir = Path(data_collection_dir)
    dataset_dir = Path(dataset_dir)

    train_img_dir = dataset_dir / "train" / "images"
    train_lbl_dir = dataset_dir / "train" / "labels"
    val_img_dir   = dataset_dir / "val" / "images"
    val_lbl_dir   = dataset_dir / "val" / "labels"

    train_img_dir.mkdir(parents=True, exist_ok=True)
    train_lbl_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir.mkdir(parents=True, exist_ok=True)
    val_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = sorted(
        p for p in data_collection_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    random.shuffle(images)
    split_idx = int(len(images) * split_percentage)

    train_images = images[:split_idx]
    val_images   = images[split_idx:]

    def copy_and_init_label(img_path, dest_img_dir, dest_lbl_dir=None):
        shutil.copy(img_path, dest_img_dir / img_path.name)
        # (dest_lbl_dir / (img_path.stem + ".txt")).touch()  # Nous pouvons soit créer des fichiers d'étiquettes vides, soit l'ignorer.

    for img in train_images:
        copy_and_init_label(img, train_img_dir, train_lbl_dir)

    for img in val_images:
        copy_and_init_label(img, val_img_dir, val_lbl_dir)

    print(f"Created train/val split: {len(train_images)} train, {len(val_images)} val")


In [ ]:
create_train_val_split(
    data_collection_dir=f"{DATA_DIR}/data_collection",
    dataset_dir=f"{DATA_DIR}/duckietown_dataset",
    split_percentage=TRAIN_TEST_SPLIT_PERCENTAGE,
)

À ce stade, vous devriez voir un dossier supplémentaire dans votre répertoire [assets/data](../../assets/data) appelé [duckietown_dataset](../../assets/data/duckietown_dataset/) qui contient la division `train` et `val` que nous avons décrite ci-dessus.


Vous êtes maintenant prêt à passer à la préparation et à l'entraînement des jeux de données. Vous pouvez continuer avec le [notebook pour entraînement](../03-Training/training.ipynb).